
las2rgb.py.ipynb
========================
Generates a Cloud-Optimized GeoTIFF (COG) RGB orthophoto from a colorized LAS point cloud.

Workflow:
  1. Sample RGB max values from the LAS to auto-detect bit depth (8-bit vs 16-bit)
  2. Run PDAL pipelines to rasterize Red, Green, Blue bands separately at 5 cm resolution
  3. Merge the three single-band GeoTIFFs into one RGB GeoTIFF
  4. Rescale to 8-bit (0–255) if source is 16-bit
  5. Convert to Cloud-Optimized GeoTIFF with LZW compression and overviews

Requirements:
  - PDAL (with Python bindings): conda install -c conda-forge pdal python-pdal
  - GDAL Python bindings:        conda install -c conda-forge gdal
  - numpy

Usage:
  python las_to_orthophoto_cog.py
  (edit INPUT_LAS and OUTPUT_DIR below, or pass --input / --output as CLI args)


In [1]:
import sys
print(sys.executable)  # tells you exactly which Python/env the kernel is using

c:\Users\tapes\anaconda3\envs\geo\python.exe


In [2]:
!conda install -c conda-forge gdal --yes --prefix {sys.prefix}

Channels:
 - conda-forge
 - defaults
 - pytorch
Platform: win-64
Solving environment: ...working... done

## Package Plan ##

  environment location: c:\Users\tapes\anaconda3\envs\geo

  added / updated specs:
    - gdal


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _libavif_api-1.4.1         |       h57928b3_0          10 KB  conda-forge
    aom-3.9.1                  |       he0c23c2_0         1.9 MB  conda-forge
    blosc-1.21.6               |       h85f69ea_0          49 KB  conda-forge
    ca-certificates-2026.4.22  |       h4c7d964_0         128 KB  conda-forge
    certifi-2026.4.22          |     pyhd8ed1ab_0         132 KB  conda-forge
    dav1d-1.2.1                |       hcfcfb64_0         604 KB  conda-forge
    fonts-conda-ecosystem-1    |                0           4 KB  conda-forge
    fonts-conda-forge-1        |       hc364b38_1           4 KB  conda-forge
    freeg

In [ ]:
"""
las_to_orthophoto_cog.py
========================
Generates a Cloud-Optimized GeoTIFF (COG) RGB orthophoto from a colorized LAS point cloud.
100% pure Python — no CLI subprocess calls required.

Workflow:
  1. Sample RGB max values from the LAS to auto-detect bit depth (8-bit vs 16-bit)
  2. Rasterize Red, Green, Blue bands via PDAL Python bindings (writers.gdal)
  3. Merge three single-band GeoTIFFs into one RGB GeoTIFF via GDAL Python API
  4. Rescale to 8-bit (0-255) if source is 16-bit
  5. Build overviews and write Cloud-Optimized GeoTIFF via GDAL Python API

Requirements (conda-forge):
  conda install -c conda-forge pdal python-pdal gdal numpy
"""

import argparse
import json
import logging
import shutil
from pathlib import Path

import numpy as np
from osgeo import gdal

gdal.UseExceptions()  # make GDAL raise Python exceptions instead of returning error codes

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG  <- edit these, or pass --input / --output as CLI args
# ──────────────────────────────────────────────────────────────────────────────
INPUT_LAS = (
    r"Y:\OMVC_GIS_Library\01_Data_Library\01_Physical_Geography"
    r"\03_LiDAR\Stinson_2BT_LiDAR\IR2_HIELLEN\03_PointCloud\cloud_merged.las"
)
OUTPUT_DIR = (
    r"Y:\OMVC_GIS_Library\01_Data_Library\01_Physical_Geography"
    r"\02_RGB_Imagery\Stinson_2BT_RGB"
)
RESOLUTION        = 0.05   # metres (5 cm)
FIRST_RETURN_ONLY = True   # True = first returns only (cleaner surface, less veg noise)
# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ── STEP 1: AUTO-DETECT RGB BIT DEPTH ────────────────────────────────────────

def detect_rgb_scale(las_path: str, sample_points: int = 500_000) -> int:
    """
    Sample up to `sample_points` from the LAS and check max RGB values.
    Returns 256 if data is 16-bit (0-65535), or 1 if already 8-bit (0-255).
    """
    import pdal

    log.info("Step 1 — Sampling %s points to detect RGB bit depth ...", f"{sample_points:,}")

    pipeline_dict = {
        "pipeline": [
            {
                "type": "readers.las",
                "filename": las_path,
                "count": sample_points,
            }
        ]
    }

    pipeline = pdal.Pipeline(json.dumps(pipeline_dict))
    pipeline.execute()
    arrays = pipeline.arrays
    if not arrays:
        raise RuntimeError("PDAL returned no data during sampling — check your LAS path.")


    log.info("  Sampled RGB max -> R: %d  G: %d  B: %d", max_r, max_g, max_b)

    if overall_max > 255:
        log.info("  -> 16-bit RGB detected (max %d). Will scale to 8-bit after merge.", overall_max)
        return 256
    else:
        log.info("  -> 8-bit RGB detected (max <= 255). No rescaling needed.")
        return 1


# ── STEP 2: RASTERIZE BANDS VIA PDAL PYTHON BINDINGS ─────────────────────────

def rasterize_band(las_path: str, out_tif: str, dimension: str,
                   resolution: float, first_return_only: bool) -> None:
    """
    Use PDAL Python bindings to rasterize one RGB band to a single-band GeoTIFF.
    writers.gdal streams the LAS in chunks — safe for large files.
    """
    import pdal

    log.info("  Rasterizing %s band -> %s", dimension, Path(out_tif).name)

    stages = [
        {
            "type": "readers.las",
            "filename": las_path,
        }
    ]

    if first_return_only:
        stages.append({
            "type": "filters.range",
            "limits": "ReturnNumber[1:1]",
        })

    stages.append({
        "type": "writers.gdal",
        "filename": out_tif,
        "dimension": dimension,
        "output_type": "mean",
        "resolution": resolution,
        "gdalopts": "COMPRESS=DEFLATE,TILED=YES,BIGTIFF=YES",
        "nodata": 0,
    })

    pipeline = pdal.Pipeline(json.dumps({"pipeline": stages}))
    pipeline.execute()
    log.info("  -> %s band done.", dimension)


# ── STEP 3: MERGE BANDS INTO RGB GEOTIFF ─────────────────────────────────────

def merge_bands(red_tif: str, green_tif: str, blue_tif: str,
                out_merged: str) -> None:
    """
    Stack three single-band GeoTIFFs into one 3-band RGB GeoTIFF using the GDAL Python API.
    Reads each source band and writes it into the output dataset in order R->G->B.
    """
    log.info("Step 3 — Merging RGB bands ...")

    ds_red   = gdal.Open(red_tif,   gdal.GA_ReadOnly)
    ds_green = gdal.Open(green_tif, gdal.GA_ReadOnly)
    ds_blue  = gdal.Open(blue_tif,  gdal.GA_ReadOnly)

    cols          = ds_red.RasterXSize
    rows          = ds_red.RasterYSize
    geo_transform = ds_red.GetGeoTransform()
    projection    = ds_red.GetProjection()
    src_dtype     = ds_red.GetRasterBand(1).DataType

    driver = gdal.GetDriverByName("GTiff")
    ds_out = driver.Create(
        out_merged, cols, rows, 3, src_dtype,
        options=["COMPRESS=DEFLATE", "TILED=YES", "BIGTIFF=YES"]
    )
    ds_out.SetGeoTransform(geo_transform)
    ds_out.SetProjection(projection)

    band_names = ["Red", "Green", "Blue"]
    for i, (ds_src, name) in enumerate(
        zip([ds_red, ds_green, ds_blue], band_names), start=1
    ):
        log.info("  Copying %s band ...", name)
        data = ds_src.GetRasterBand(1).ReadAsArray()
        ds_out.GetRasterBand(i).WriteArray(data)
        ds_out.GetRasterBand(i).SetNoDataValue(0)

    ds_red = ds_green = ds_blue = None
    ds_out.FlushCache()
    ds_out = None
    log.info("  -> Bands merged: %s", Path(out_merged).name)


# ── STEP 4: RESCALE TO 8-BIT ──────────────────────────────────────────────────

def rescale_to_8bit(src_tif: str, dst_tif: str, scale_divisor: int) -> None:
    """
    If data is 16-bit (scale_divisor=256), scale each band from [0,65535]->[0,255]
    and write a new Byte-type GeoTIFF. Otherwise just copy the file.
    """
    if scale_divisor == 1:
        log.info("Step 4 — RGB already 8-bit, skipping rescale ...")
        shutil.copy2(src_tif, dst_tif)
        return

    log.info("Step 4 — Rescaling 16-bit -> 8-bit ...")

    ds_src        = gdal.Open(src_tif, gdal.GA_ReadOnly)
    cols          = ds_src.RasterXSize
    rows          = ds_src.RasterYSize
    geo_transform = ds_src.GetGeoTransform()
    projection    = ds_src.GetProjection()
    n_bands       = ds_src.RasterCount

    driver = gdal.GetDriverByName("GTiff")
    ds_dst = driver.Create(
        dst_tif, cols, rows, n_bands, gdal.GDT_Byte,
        options=["COMPRESS=DEFLATE", "TILED=YES", "BIGTIFF=YES"]
    )
    ds_dst.SetGeoTransform(geo_transform)
    ds_dst.SetProjection(projection)

    for b in range(1, n_bands + 1):
        log.info("  Rescaling band %d/%d ...", b, n_bands)
        data        = ds_src.GetRasterBand(b).ReadAsArray().astype(np.float32)
        nodata_mask = (data == 0)
        scaled      = np.clip(data / 65535.0 * 255.0, 0, 255).astype(np.uint8)
        scaled[nodata_mask] = 0  # preserve nodata pixels
        ds_dst.GetRasterBand(b).WriteArray(scaled)
        ds_dst.GetRasterBand(b).SetNoDataValue(0)

    ds_src = None
    ds_dst.FlushCache()
    ds_dst = None
    log.info("  -> Rescaled: %s", Path(dst_tif).name)


# ── STEP 5: BUILD OVERVIEWS + WRITE COG ──────────────────────────────────────

def convert_to_cog(src_tif: str, cog_tif: str) -> None:
    """
    Build internal overviews on the source then write a Cloud-Optimized GeoTIFF
    using GDAL's COG driver — all via the Python API.
    """
    log.info("Step 5 — Building overviews ...")
    ds = gdal.Open(src_tif, gdal.GA_Update)
    overview_levels = [2, 4, 8, 16, 32, 64, 128]
    ds.BuildOverviews("AVERAGE", overview_levels)
    ds = None
    log.info("  -> Overviews built: %s", overview_levels)

    log.info("  Writing Cloud-Optimized GeoTIFF ...")
    ds_src = gdal.Open(src_tif, gdal.GA_ReadOnly)

    translate_options = gdal.TranslateOptions(
        format="COG",
        creationOptions=[
            "COMPRESS=LZW",
            "PREDICTOR=2",
            "BIGTIFF=YES",
            "RESAMPLING=AVERAGE",
            "OVERVIEWS=IGNORE_EXISTING",
        ],
    )
    gdal.Translate(cog_tif, ds_src, options=translate_options)
    ds_src = None
    log.info("  -> COG written: %s", Path(cog_tif).name)


# ── STEP 6: VALIDATE COG ──────────────────────────────────────────────────────

def validate_cog(cog_tif: str) -> None:
    """Validate COG structure using GDAL's Python utility."""
    try:
        from osgeo.utils.validate_cloud_optimized_geotiff import validate
        is_valid, errors, warnings = validate(cog_tif, full_check=False)
        if is_valid:
            log.info("Step 6 — COG validation PASSED")
        else:
            log.warning("Step 6 — COG validation issues: %s", errors)
        if warnings:
            log.warning("  COG warnings: %s", warnings)
    except ImportError:
        log.info("Step 6 — COG validation skipped (osgeo.utils not available). Output is still valid.")


# ── PRINT OUTPUT INFO ─────────────────────────────────────────────────────────

def print_raster_info(tif_path: str) -> None:
    """Print basic raster metadata using GDAL Python API."""
    try:
        ds = gdal.Open(tif_path, gdal.GA_ReadOnly)
        gt = ds.GetGeoTransform()
        log.info("─" * 60)
        log.info("Output raster info:")
        log.info("  Size      : %d cols x %d rows x %d bands",
                 ds.RasterXSize, ds.RasterYSize, ds.RasterCount)
        log.info("  Pixel size: %.4f m x %.4f m", abs(gt[1]), abs(gt[5]))
        log.info("  Origin    : (%.4f, %.4f)", gt[0], gt[3])
        log.info("  Projection: %s ...", ds.GetProjection()[:80])
        for b in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(b)
            mn, mx, mean, std = band.GetStatistics(True, True)
            log.info("  Band %d    : min=%.0f  max=%.0f  mean=%.1f  std=%.1f",
                     b, mn, mx, mean, std)
        ds = None
    except Exception as e:
        log.warning("Could not read output info: %s", e)


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="LAS RGB -> Cloud-Optimized GeoTIFF (pure Python)")
    parser.add_argument("--input",       default=INPUT_LAS,   help="Path to input .las file")
    parser.add_argument("--output",      default=OUTPUT_DIR,  help="Output directory")
    parser.add_argument("--resolution",  type=float, default=RESOLUTION,
                        help="Raster resolution in metres (default: 0.05 = 5 cm)")
    parser.add_argument("--all-returns", action="store_true",
                        help="Use all returns (default: first-return only)")
    args, _ = parser.parse_known_args()  # ignore Jupyter's --f= kernel arg

    las_path          = args.input
    output_dir        = Path(args.output)
    resolution        = args.resolution
    first_return_only = not args.all_returns

    output_dir.mkdir(parents=True, exist_ok=True)

    stem      = Path(las_path).stem
    res_label = f"{int(resolution * 100)}cm"

    # Intermediate files (prefixed _tmp_ so they're easy to identify/delete)
    red_tif    = str(output_dir / f"_tmp_{stem}_RED_{res_label}.tif")
    green_tif  = str(output_dir / f"_tmp_{stem}_GREEN_{res_label}.tif")
    blue_tif   = str(output_dir / f"_tmp_{stem}_BLUE_{res_label}.tif")
    merged_tif = str(output_dir / f"_tmp_{stem}_RGB_merged_{res_label}.tif")
    scaled_tif = str(output_dir / f"_tmp_{stem}_RGB_8bit_{res_label}.tif")
    cog_tif    = str(output_dir / f"{stem}_RGB_{res_label}_COG.tif")

    log.info("=" * 60)
    log.info("LAS -> RGB Orthophoto COG")
    log.info("Input     : %s", las_path)
    log.info("Output    : %s", cog_tif)
    log.info("Resolution: %.2f m  |  First-return filter: %s", resolution, first_return_only)
    log.info("=" * 60)

    scale_divisor = detect_rgb_scale(las_path)

    log.info("Step 2 — Rasterizing RGB bands (slow step for a 20 GB file) ...")
    for dimension, out_tif in [("Red", red_tif), ("Green", green_tif), ("Blue", blue_tif)]:
        rasterize_band(las_path, out_tif, dimension, resolution, first_return_only)

    merge_bands(red_tif, green_tif, blue_tif, merged_tif)
    rescale_to_8bit(merged_tif, scaled_tif, scale_divisor)
    convert_to_cog(scaled_tif, cog_tif)
    validate_cog(cog_tif)

    log.info("Cleaning up temporary files ...")
    for tmp in [red_tif, green_tif, blue_tif, merged_tif, scaled_tif]:
        try:
            Path(tmp).unlink(missing_ok=True)
        except Exception:
            pass

    log.info("=" * 60)
    log.info("DONE -> %s", cog_tif)
    print_raster_info(cog_tif)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'osgeo'